# scRNA-seq 분석 실습

건국대학교 이형우 교수님 연구실 온라인 세미나
Data: GSE210543 — Human Retina (Young vs Old)
Runtime: Google Colab + Seurat v5 (R)

## 0. 데이터셋 개요 — GSE210543

### Cell Ranger Web Summary

10X Genomics Cell Ranger가 시퀀싱 완료 후 생성하는 QC 리포트입니다.
분석 시작 전 반드시 확인해야 하는 3가지 지표:

| 지표 | 의미 | 권장 기준 |
|------|------|---------|
| **Estimated Number of Cells** | Cell Ranger 추정 세포 수 | > 2,000 |
| **Median Genes per Cell** | 세포당 검출 유전자 중앙값 | > 1,500 |
| **Fraction Reads in Cells** | 세포에 할당된 reads 비율 | > 70% |

> **💡 Tip:** Mean Reads per Cell이 매우 높다면 세포 수가 너무 적어서 생기는 수학적 효과일 수 있습니다 — 단독으로 해석하지 마세요.

### 전체 13개 샘플 품질 요약

| 샘플 | 그룹 | 세포 수 | Mean Reads/Cell | Median Genes/Cell | 선택 |
|------|------|-------:|---------------:|------------------:|:----:|
| **16PCW** | Young | **8,601** | 43,243 | 1,084 | ✅ |
| **20PCW** | Young | **5,787** | 103,036 | **5,174** | ✅ |
| 12PCW | Young | 3,637 | 164,166 | 4,596 | — |
| 21PCW | Young | 3,948 | 139,139 | 1,953 | — |
| **Adult_2** | Old | **6,516** | 90,897 | 2,016 | ✅ |
| **Adult_3** | Old | **3,694** | 142,303 | 2,806 | ✅ |
| Adult_5 | Old | 3,487 | 168,631 | 2,529 | — |
| Adult_1 | Old | 884 | 412,206 | 2,807 | ❌ |
| Adult_4 | Old | 496 | 1,146,570 | 2,414 | ❌ |
| AMD_macula | AMD | 1,719 | 284,749 | 4,002 | — |
| AMD_peripheral | AMD | 1,794 | 267,813 | 3,584 | — |
| Unaffected_macula | Control | 3,557 | 101,143 | 5,102 | — |
| Unaffected_peripheral | Control | 3,527 | 112,731 | 4,765 | — |

✅ 세미나 분석 샘플 (Young 2 + Old 2) | ❌ 세포 수 부족 또는 품질 이슈

### Web Summary 읽는 법 — 좋은 샘플 vs 나쁜 샘플

#### ✅ 좋은 샘플 — 20PCW

<table width="100%" cellpadding="0" cellspacing="0" style="background: #F0FDF4; border: 2px solid #16A34A; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #16A34A; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">GOOD</span>
    <span style="font-size: 17px; font-weight: 700; color: #14532D;">20PCW — 이상적인 라이브러리</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #166534; font-size: 13px; white-space: nowrap; vertical-align: middle;">5,787 cells &middot; 5,174 genes/cell &middot; 91.3% in cells</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_good_20PCW.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **5,787** | ✅ 충분 |
| Mean Reads/Cell | 103,036 | ✅ 정상 범위 |
| Median Genes/Cell | **5,174** | ✅ 매우 우수 |
| Fraction in Cells | **91.3%** | ✅ 배경 노이즈 최소 |

Barcode Rank Plot에서 파란 선과 회색 선 사이의 **꺾임(knee)**이 뚜렷합니다.

---

#### ❌ 나쁜 샘플 — Adult_4

<table width="100%" cellpadding="0" cellspacing="0" style="background: #FFF1F2; border: 2px solid #E11D48; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #E11D48; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">BAD</span>
    <span style="font-size: 17px; font-weight: 700; color: #881337;">Adult_4 — 세포 포획 실패</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #BE123C; font-size: 13px; white-space: nowrap; vertical-align: middle;">496 cells &middot; 1,146,570 reads/cell</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_bad_Adult4.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **496** | ❌ 세포 포획 실패 |
| Mean Reads/Cell | **1,146,570** | ⚠️ 비정상 (수학적 효과) |
| Median Genes/Cell | 2,414 | ⚠️ 보통 |
| Fraction in Cells | 78.8% | ⚠️ 낮은 편 |

```
Mean Reads/Cell = 총 reads ÷ 세포 수
               = 568,698,782 ÷ 496 ≈ 1,146,570
```

→ 총 reads는 20PCW(596M)와 비슷하지만 세포 수가 극히 적어 값이 폭등.
**세포 포획 실패**가 원인 — 분석에서 제외.

## Step 0. 환경 설정

> **Colab 설정:** Runtime → Change runtime type → **R**

패키지 설치는 처음 1번만 실행합니다 (~10분 소요).

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_04.png" width="850"/>

*Fig. 0 — scRNA-seq 분석의 복잡성 (Hicks et al., BioRxiv 2015)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_05.png" width="850"/>

*Fig. 1 — scRNA-seq 전체 분석 워크플로우 (Luecken & Theis, Mol Syst Biol 2019)*

In [ ]:
# 필요 패키지 설치 (처음 1회만 실행 — 약 10분 소요)
pkgs <- c("Seurat", "harmony", "dplyr", "ggplot2", "patchwork", "clustree", "plyr")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p)
}

In [ ]:
library(Seurat)
library(harmony)
library(dplyr)
library(ggplot2)
library(patchwork)

set.seed(42)
cat("Seurat:", as.character(packageVersion("Seurat")), "\n")
cat("harmony:", as.character(packageVersion("harmony")), "\n")

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1*

## Step 1. 데이터 불러오기

10X Genomics `filtered_feature_bc_matrix` 폴더를 읽어 Seurat 오브젝트를 생성합니다.

```
filtered_feature_bc_matrix/
├── barcodes.tsv.gz   ← 세포 바코드
├── features.tsv.gz   ← 유전자 ID/이름
└── matrix.mtx.gz     ← UMI count matrix (sparse)
```

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_08.png" width="850"/>

*Fig. 4 — Count Matrix 생성 원리: barcodes / features / matrix (Macosko et al., Cell 2015)*

In [ ]:
# Google Drive 마운트
# Colab 좌측 파일 아이콘 → Drive 마운트 버튼 클릭
# 또는 아래 실행 후 인증 링크 클릭

# R에서 Drive 마운트 명령 실행
system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# 데이터 경로 설정 (공유된 Google Drive 경로로 수정 필요)
DATA_DIR <- "/content/drive/MyDrive/KU_seminar/GSE210543/filtered_feature_bc_matrix"
cat("Data directory:", DATA_DIR, "\n")

In [ ]:
# 4개 샘플 불러오기
samples <- list(
  Young_16PCW  = file.path(DATA_DIR, "16PCW"),
  Young_20PCW  = file.path(DATA_DIR, "20PCW"),
  Old_Adult2   = file.path(DATA_DIR, "Adult_2"),
  Old_Adult3   = file.path(DATA_DIR, "Adult_3")
)

# Seurat 오브젝트 생성
seurat_list <- lapply(names(samples), function(name) {
  cat("Loading:", name, "...\n")
  counts <- Read10X(data.dir = samples[[name]])
  obj <- CreateSeuratObject(
    counts  = counts,
    project = name,
    min.cells = 3,    # 최소 3개 세포에서 발현된 유전자만 포함
    min.features = 200  # 최소 200개 유전자가 발현된 세포만 포함
  )
  obj$sample <- name
  obj$group  <- ifelse(grepl("Young", name), "Young", "Old")
  return(obj)
})
names(seurat_list) <- names(samples)

# 각 샘플 기본 정보 확인
for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat(sprintf("%s: %d cells, %d genes\n", name, ncol(obj), nrow(obj)))
}

In [ ]:
# 4개 샘플 불러오기
samples <- list(
  Young_16PCW  = file.path(DATA_DIR, "16PCW"),
  Young_20PCW  = file.path(DATA_DIR, "20PCW"),
  Old_Adult2   = file.path(DATA_DIR, "Adult_2"),
  Old_Adult3   = file.path(DATA_DIR, "Adult_3")
)

# Seurat 오브젝트 생성
seurat_list <- lapply(names(samples), function(name) {
  cat("Loading:", name, "...\n")
  counts <- Read10X(data.dir = samples[[name]])
  obj <- CreateSeuratObject(
    counts  = counts,
    project = name,
    min.cells = 3,    # 최소 3개 세포에서 발현된 유전자만 포함
    min.features = 200  # 최소 200개 유전자가 발현된 세포만 포함
  )
  obj$sample <- name
  obj$group  <- ifelse(grepl("Young", name), "Young", "Old")
  return(obj)
})
names(seurat_list) <- names(samples)

# 각 샘플 기본 정보 확인
for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat(sprintf("%s: %d cells, %d genes\n", name, ncol(obj), nrow(obj)))
}

## Step 2. QC — Quality Control

### 핵심 QC 지표 3가지

| 지표 | 의미 | 기본 필터 |
|------|------|---------|
| `nFeature_RNA` | 세포당 검출 유전자 수 | > 200 |
| `nCount_RNA` | 세포당 총 UMI 수 | > 500 |
| `percent.mt` | 미토콘드리아 유전자 비율 | < 10% |

**낮은 nFeature** → 빈 droplet 또는 죽은 세포
**높은 percent.mt** → 세포막 손상, 세포질 RNA 유출

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_09.png" width="850"/>

*Fig. 5 — QC 지표 분포 확인 방법 (NCells, nUMI, nGene, mitoRatio)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_10.png" width="850"/>

*Fig. 6 — QC VlnPlot 예시: nFeature_RNA / nCount_RNA / percent.mt*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_11.png" width="850"/>

*Fig. 7 — QC 필터링 실습 코드 및 Tip 2: Filter 조건에서의 정답은 없다!*

In [ ]:
# 미토콘드리아 유전자 비율 계산
# 인간 미토콘드리아 유전자는 'MT-' 로 시작
seurat_list <- lapply(seurat_list, function(obj) {
  obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
  return(obj)
})

# QC 지표 확인 (첫 번째 샘플 예시)
head(seurat_list[[1]]@meta.data[, c("nFeature_RNA", "nCount_RNA", "percent.mt")])

In [ ]:
# nFeature vs nCount 산점도 — doublet 탐지
# 정상 세포는 선형 관계를 보임
# 이상치(doublet)는 오른쪽 상단에 위치

scatter_list <- lapply(names(seurat_list), function(name) {
  p1 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "percent.mt"
  ) + ggtitle(paste(name, "- Count vs MT%"))

  p2 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "nFeature_RNA"
  ) + ggtitle(paste(name, "- Count vs Feature"))

  p1 + p2
})

for (p in scatter_list) print(p)

## Step 3. QC 필터링

```r
nFeature_RNA > 200  &  nCount_RNA > 500  &  percent.mt < 10
```

> **💡 Tip:** 임계값은 샘플마다 다릅니다. VlnPlot을 보고 분포의 자연스러운 경계(valley)를 찾으세요.

In [ ]:
# QC 기준값 설정 (여기서 조정 가능)
QC_MIN_FEATURE <- 200   # 최소 유전자 수
QC_MIN_COUNT   <- 500   # 최소 UMI 수
QC_MAX_MT      <- 10    # 최대 미토콘드리아 비율 (%)

# 필터링 적용
seurat_list_filtered <- lapply(names(seurat_list), function(name) {
  obj <- seurat_list[[name]]
  before <- ncol(obj)

  obj <- subset(
    obj,
    subset = nFeature_RNA > QC_MIN_FEATURE &
             nCount_RNA   > QC_MIN_COUNT   &
             percent.mt   < QC_MAX_MT
  )

  after <- ncol(obj)
  removed <- before - after
  cat(sprintf("%s: %d → %d cells (removed %d, %.1f%%)\n",
              name, before, after, removed, removed/before*100))
  return(obj)
})
names(seurat_list_filtered) <- names(seurat_list)

## Step 4. 정규화 — Normalization

세포마다 시퀀싱 깊이가 다릅니다. 정규화로 이를 보정합니다.

**LogNormalize** (기본값):
```
normalized = log(count / total_count × 10,000 + 1)
```

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_12.png" width="850"/>

*Fig. 8 — LogNormalization vs SCTransform 방법 비교 및 PC 선택 기준 (Tip 3)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_13.png" width="850"/>

*Fig. 9 — 정규화 개념: SCTransform vs LogNormalization (M. Loven, RNA-seq statistical analysis)*

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  NormalizeData(
    obj,
    normalization.method = "LogNormalize",
    scale.factor = 10000
  )
})

cat("Normalization complete!\n")

## Step 5. HVG — 고변이 유전자 선택

전체 수만 개 유전자 중 **샘플 간 차이가 큰 유전자**만 선택하여 분석 효율을 높입니다.

일반적으로 상위 2,000개 사용 (Seurat 기본값).

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  FindVariableFeatures(
    obj,
    selection.method = "vst",
    nfeatures = 2000
  )
})

# Top 10 HVG 확인 (첫 번째 샘플)
top10 <- head(VariableFeatures(seurat_list_filtered[[1]]), 10)
cat("Top 10 HVGs (16PCW):\n")
print(top10)

---

## 💾 중간 체크포인트

QC + 정규화 + HVG까지 완료했습니다. 오브젝트를 저장합니다.

> **💡 Tip:** 세션이 끊기면 Step 0 라이브러리 로드 후 아래 **로드 셀**을 실행하세요.

## Step 6. Cell Cycle Scoring *(Optional)*

세포 주기(G1/S/G2M)가 클러스터링에 영향을 줄 수 있습니다.
발달기(Young) 샘플에는 증식 세포가 많으므로 확인이 필요합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_14.png" width="850"/>

*Fig. 10 — Cell Cycle Scoring: 언제 회귀할지, 언제 남겨둘지*

In [ ]:
# Cell cycle 관련 유전자 (Seurat 내장)
s.genes   <- cc.genes$s.genes    # S phase
g2m.genes <- cc.genes$g2m.genes  # G2M phase

# Cell cycle score 계산
seurat_merged <- CellCycleScoring(
  seurat_merged,
  s.features   = s.genes,
  g2m.features = g2m.genes,
  set.ident    = TRUE
)

# 분포 확인
table(seurat_merged$Phase)

## Step 7. Scaling + PCA

**Scaling**: 유전자별 평균 0, 분산 1로 맞춰 발현량 차이를 제거합니다.
**PCA**: 고변이 유전자 2,000개 → 주요 성분 50개로 차원 축소.

In [ ]:
# Scaling (cell cycle 회귀 포함)
seurat_merged <- ScaleData(
  seurat_merged,
  vars.to.regress = c("S.Score", "G2M.Score"),  # cell cycle 보정
  features        = rownames(seurat_merged)
)

cat("Scaling complete!\n")

In [ ]:
# PCA 실행
seurat_merged <- RunPCA(
  seurat_merged,
  features = VariableFeatures(seurat_merged),
  npcs     = 50
)

# Elbow Plot — 몇 개의 PC를 사용할지 결정
ElbowPlot(seurat_merged, ndims = 50) +
  ggtitle("Elbow Plot: PC 기여도") +
  geom_vline(xintercept = 30, linetype = "dashed", color = "red") +
  annotate("text", x = 32, y = 3, label = "PC=30 선택", color = "red")

## Step 8. Integration — Harmony

Young(발달기)과 Old(성체) 샘플은 생물학적 차이 외에도 배치 효과(batch effect)가 있습니다.
**Harmony**로 배치를 보정하되 생물학적 신호는 보존합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_15.png" width="850"/>

*Fig. 11 — Integration 방법 비교: Harmony가 세포 타입 구조를 잘 보존 (Tran et al., Genome Biol 2020)*

## Step 9. UMAP

고차원 데이터를 2D로 시각화합니다.
Harmony 보정 결과(`reduction = "harmony"`)를 입력으로 사용합니다.

In [ ]:
# UMAP (Harmony 통합 결과 기반)
seurat_merged <- RunUMAP(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 통합 전/후 비교
p_before <- DimPlot(seurat_merged, reduction = "pca",  group.by = "sample") + ggtitle("Before Integration (PCA)")
p_after  <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample") + ggtitle("After Integration (UMAP)")
p_before + p_after

## Step 10. Clustering

KNN 그래프 기반 Louvain/Leiden 클러스터링입니다.
`resolution`이 클수록 더 많은 클러스터가 생성됩니다.

> **💡 Tip:** Resolution 0.4~0.6에서 시작하세요. VlnPlot과 마커 유전자를 보고 최종 선택합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_16.png" width="850"/>

*Fig. 12 — Resolution에 따른 클러스터링 결과 비교 (Res 0.2 ~ 1.2)*

In [ ]:
# KNN 그래프 생성
seurat_merged <- FindNeighbors(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 여러 Resolution으로 클러스터링
resolutions <- c(0.2, 0.4, 0.6, 0.8)
for (res in resolutions) {
  seurat_merged <- FindClusters(
    seurat_merged,
    resolution   = res,
    cluster.name = paste0("RNA_snn_res.", res)
  )
  cat(sprintf("Res %.1f: %d clusters\n", res, length(unique(seurat_merged@meta.data[[paste0("RNA_snn_res.", res)]])))  )
}

In [ ]:
# 최적 Resolution 선택 (세미나: 0.4 사용)
Idents(seurat_merged) <- "RNA_snn_res.0.4"
seurat_merged$seurat_clusters <- Idents(seurat_merged)

DimPlot(seurat_merged, reduction = "umap", label = TRUE, label.size = 4) +
  ggtitle("Final Clustering (Resolution 0.4)") +
  theme_minimal()

## Step 11. Marker Gene Identification

각 클러스터를 나머지 클러스터와 비교하여 대표 유전자를 찾습니다.

`FindAllMarkers()` 주요 파라미터:
- `only.pos = TRUE`: 해당 클러스터에서 높게 발현되는 유전자만
- `min.pct = 0.25`: 최소 25% 세포에서 발현
- `logfc.threshold = 0.25`: 최소 log2FC 기준

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_17.png" width="850"/>

*Fig. 13 — 세포 타입 어노테이션 전략: 자동 어노테이션 vs 수동 어노테이션 (Clake ZA et al., Nat Protoc 2019)*

In [ ]:
# 마커 유전자 탐색 (시간 소요: 5~15분)
# only.pos = TRUE: 해당 클러스터에서 높게 발현되는 유전자만
markers <- FindAllMarkers(
  seurat_merged,
  only.pos          = TRUE,
  min.pct           = 0.25,  # 최소 25% 세포에서 발현
  logfc.threshold   = 0.25   # 최소 log2FC 0.25
)

# Top 5 마커 확인
markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 5) %>%
  print(n = Inf)

In [ ]:
# Top 10 마커 히트맵
top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 10)

DoHeatmap(seurat_merged, features = top10$gene) +
  theme(axis.text.y = element_text(size = 6))

## Step 12. Cell Type Annotation

### 마커 유전자 — Voigt et al. 2022 (PMC10162434) 기반

| 세포 타입 | 마커 유전자 | 비고 |
|----------|------------|------|
| RGC | SNCG, ISL1, POU4F2 | BRN3 계열 + SNCG |
| Amacrine | TFAP2A, PAX6, GAD1 | — |
| Bipolar | CABP5, PRKCA, GRM6 | VSX2 제외 (Progenitor 혼용 위험) |
| Müller glia | GLUL, RLBP1, SLC1A3 | 3개 모두 고신뢰 마커 |
| Rod | RHO, NRL, RCVRN | NRL = rod master TF |
| Cone | OPN1LW, ARR3, GNGT2 | — |
| Horizontal | LHX1, ONECUT2, PROX1 | — |
| Progenitor | VSX2, FGF19, LIN28B | Young 샘플에 풍부 |
| Microglia | CX3CR1, P2RY12, TMEM119 | — |
| Endothelial | PECAM1, CDH5, VWF | — |

> **💡 Tip:** Bipolar에서 VSX2를 제거했습니다 — 발달기 Progenitor와 혼용 위험이 있어 CABP5/PRKCA로 구별합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_18.png" width="850"/>

*Fig. 14 — 세포 타입 어노테이션 방법 및 Tip 4: 연구자의 주관이 중요!*

### Task. 어노테이션 테스트 — 3개 세포 타입

논문에서 **마커 특이성 + 풍부도** 기준으로 선별:

| 세포 타입 | 마커 | 선별 이유 |
|----------|------|---------|
| **Müller glia** | GLUL, RLBP1, SLC1A3 | 3개 모두 독립적으로 Müller를 정의 |
| **Rod** | RHO, NRL, RCVRN | 성체 망막 최다 세포 + NRL은 rod master TF |
| **RGC** | SNCG, ISL1, POU4F2 | POU4F(BRN3) + SNCG 조합 = best marker set |

In [ ]:
# ── 논문 기반 마커 설정 (Voigt et al. 2022 / PMC10162434) ──────
# 3개 세포 타입 선별: Müller glia / Rod / RGC
selected_markers <- list(
  "Muller_glia" = c("GLUL", "RLBP1", "SLC1A3"),
  "Rod"         = c("RHO",  "NRL",   "RCVRN"),
  "RGC"         = c("SNCG", "ISL1",  "POU4F2")
)

# 데이터에 실제 존재하는 유전자만 필터링
selected_markers_valid <- lapply(selected_markers, function(genes) {
  found <- genes[genes %in% rownames(seurat_merged)]
  missing <- genes[!genes %in% rownames(seurat_merged)]
  if (length(missing) > 0)
    cat(sprintf("[주의] 데이터에 없는 유전자: %s\n", paste(missing, collapse=", ")))
  found
})
selected_markers_valid <- selected_markers_valid[sapply(selected_markers_valid, length) > 0]
all_sel <- unlist(selected_markers_valid)

cat("\n최종 사용 마커 목록:\n")
for (ct in names(selected_markers_valid)) {
  cat(sprintf("  %-15s: %s\n", ct, paste(selected_markers_valid[[ct]], collapse=", ")))
}

In [ ]:
# 클러스터 → 세포 타입 매핑 (분석 결과 보고 수정)
# 아래는 예시 - 실제 마커 확인 후 조정 필요
cluster_annotations <- c(
  "0"  = "Muller_glia",
  "1"  = "Progenitor",
  "2"  = "Bipolar",
  "3"  = "RGC",
  "4"  = "Amacrine",
  "5"  = "Rod",
  "6"  = "Cone",
  "7"  = "Horizontal",
  "8"  = "Microglia",
  "9"  = "Unknown"
)

seurat_merged$cell_type <- plyr::mapvalues(
  as.character(seurat_merged$seurat_clusters),
  from = names(cluster_annotations),
  to   = cluster_annotations
)

# 최종 UMAP
DimPlot(seurat_merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, label.size = 3, repel = TRUE) +
  ggtitle("Cell Type Annotation") +
  theme_minimal()

In [ ]:
# Young vs Old: 세포 타입 구성 비교
prop_df <- seurat_merged@meta.data %>%
  group_by(group, cell_type) %>%
  summarise(n = n(), .groups = "drop") %>%
  group_by(group) %>%
  mutate(proportion = n / sum(n))

ggplot(prop_df, aes(x = group, y = proportion, fill = cell_type)) +
  geom_bar(stat = "identity") +
  scale_fill_brewer(palette = "Set3") +
  labs(title = "Cell Type Composition: Young vs Old",
       x = "Group", y = "Proportion") +
  theme_minimal()

In [ ]:
# 최종 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "02_seurat_annotated.rds"))
cat("Saved: 02_seurat_annotated.rds\n")

# 요약
cat("\n=== Final Summary ===\n")
cat("Total cells:", ncol(seurat_merged), "\n")
cat("Cell types:\n")
print(table(seurat_merged$cell_type, seurat_merged$group))

---

## 🎉 세미나 완료

| 단계 | 완료 내용 |
|------|---------|
| Step 0 | 환경 설정 (Seurat, Harmony) |
| Step 1 | 10X 데이터 로드 (4개 샘플) |
| Step 2-3 | QC 시각화 + 필터링 |
| Step 4-5 | LogNormalize + HVG |
| Step 6 | Cell Cycle (optional) |
| Step 7 | Scaling + PCA |
| Step 8 | Harmony 통합 |
| Step 9 | UMAP |
| Step 10 | Clustering |
| Step 11 | FindAllMarkers |
| Step 12 | Cell Type Annotation |